## Feature Engineering
 - Feature Engineeering means preparing the data so that the ML algorithm can Learn Effectively.
 
### Objective

The objective of this notebook is to transform the ML-ready dataset into a format suitable for machine learning by:

- Removing unnecessary columns
- Handling missing values
- Preventing data leakage
- Encoding categorical features
- Creating the final feature vector
- Preparing data for model training

### Step 1 : Load the ML Dataset After EDA


In [0]:
feature_df = spark.read.table('retail_project.gold.ml_sales_data_aftereda')

display(feature_df)

In [0]:
#Checking the shape of Rows and Columns
print('Rows after EDA :', feature_df.count())
print('Columns after EDA :', len(feature_df.columns))

#### Step 2 — Verify Schema Again

In [0]:
feature_df.printSchema()

### Step 3 — Identify the Target Variable

In [0]:
# First data scientist we know the target column what we are predicting?

target_feature = 'Total_revenue'

#### Step 4- Removing unnecessary columns for model building


In [0]:
# Removing Year Column because the dataset contain only 2025 data

feature_df = feature_df.drop('Year')


#### step 5: Create Date Features

- Instead of using the raw `order_date`, derive features that models can learn from.

In [0]:
from pyspark.sql.functions import (
    quarter, dayofweek, dayofmonth, weekofyear
)

feature_df = feature_df\
    .withColumn('Quarter', quarter(feature_df['order_date']))\
    .withColumn('DayOfWeek', dayofweek(feature_df['order_date']))\
    .withColumn('DayOfMonth', dayofmonth(feature_df['order_date']))\
    .withColumn('WeekOfYear', weekofyear(feature_df['order_date']))

display(feature_df)


## Why Create These Features? 

Because,

| **Feature**    | **Purpose**                                                              |
| -------------- | ------------------------------------------------------------------------ |
| **Quarter**    | Captures seasonal business trends across the year.                       |
| **DayOfWeek**  | Identifies differences in weekday and weekend purchasing behavior.       |
| **WeekOfYear** | Helps analyze weekly sales and demand patterns.                          |
| **DayOfMonth** | Captures monthly trends, such as salary cycles and month-end purchasing. |

Although the current dataset covers only **three months**, creating these date-based features follows **production best practices**. It improves model scalability and ensures the pipeline can handle larger datasets without requiring additional changes.


In [0]:
# Now dropping the original 'order_date' column and 'Total_orders' column because it as almost constant value 1 and not useful for our model
feature_df = feature_df.drop('order_date', 'Total_orders')

display(feature_df)

In [0]:
x = feature_df.select('product_id', 'product_name').distinct().orderBy('product_id')
display(x)

* Each product has a unique **`product_id`**, which uniquely identifies the product. Therefore, we can safely drop the **`product_name`** feature, as the model can use **`product_id`** instead of the product name for training.


In [0]:
# We can also drop the 'product_name' column as it is not needed for our model because im model train we use only the product_id

feature_df = feature_df.drop('product_name')

display(feature_df)

In [0]:
feature_df.printSchema()

#### Why Can't We Train the Model Directly?

- The dataset still contains **categorical (text) features**, while machine learning algorithms can only process **numerical data**.

- Therefore, before training the model, we must **convert all categorical features into numerical values** using appropriate encoding techniques.


In [0]:
# checking Rows and columns 
print('Shape of the Dataset:', (feature_df.count(), len(feature_df.columns)))

#### PySpark ML Feature Engineering Pipeline

We'll build this pipeline step by step.

    Categorical Columns
            │
            ▼
    StringIndexer
            │
            ▼
    Indexed Columns
            │
            ▼
    OneHotEncoder
            │
            ▼
    Encoded Columns
            │
            ▼
    VectorAssembler
            │
            ▼
    features Vector
            │
            ▼
    Regression Model

In [0]:
feature_df.printSchema()

##### Step 1 – Identify Feature Types

In [0]:
categorical_cols = [
    "product_id",
    "category",
    "gender",
    "city",
    "state"
]

numeric_cols = [
    "Month",
    "Quarter",
    "WeekOfYear",
    "DayOfWeek",
    "DayOfMonth",
    "Total_quantity",
    "Max_price"
]


- Different preprocessing is required for different data types.

##### Step 2 – Checking Missing Values

In [0]:
from pyspark.sql.functions import col, count, when

display(
    feature_df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in feature_df.columns
    ])
)

- No Missing Values Present.

##### Step 3 – StringIndexer

In [0]:
from pyspark.ml.feature import StringIndexer

indexers = [
    StringIndexer(
        inputCol= col_name,
        outputCol= col_name + '_index',
        handleInvalid = 'keep'
    )
    for col_name in categorical_cols
]

### **Short Inference**

* **`handleInvalid="keep"` prevents prediction failures by assigning unseen categories to a separate bucket.**
* **It makes the ML pipeline robust and production-ready by handling new categories gracefully.**
- Example: During Training we have 
        - Electronics
        - Fashion
        - Grocery
- After Deploying we add 
        - Sports

Without handleInvalid="keep":
- ❌ Your model throws an error.

With it:

- ✅ Spark assigns unseen values to a special bucket and continues working.

Always use it unless you have a specific reason not to.

##### Step 4 – OneHotEncoder

In [0]:
from pyspark.ml.feature import OneHotEncoder

encoder = OneHotEncoder(
    inputCols = [c + '_index' for c in categorical_cols],
    outputCols= [c + '_encoded' for c in categorical_cols]
)

#### **Why not use the indexed values directly?**

* **Indexed values can introduce a false numerical order between categories.**
* **One-Hot Encoding eliminates this by representing each category as an independent binary feature, making it suitable for nominal data.**


##### Step 5 – Verify the both StringIndexer and OneHot Encoding

- For learning: ✅ Verify after Step 4.

- For a production project: ⏩ You can move directly to VectorAssembler and then build the complete pipeline.

#### How do we verify?

- Instead of building the complete pipeline immediately, let's test the transformations one by one.


##### Step 1: Fit and Transform with StringIndexer

In [0]:
from pyspark.ml.feature import StringIndexer

# Example for category column
category_indexer = StringIndexer(
    inputCol="category",
    outputCol="category_index",
    handleInvalid="keep"
)

# Fit the indexer
category_model = category_indexer.fit(feature_df)

# Transform the DataFrame
indexed_df = category_model.transform(feature_df)

In [0]:
# Display the Indexed DataFrame
display(indexed_df.select("category","category_index").distinct())

- Now you can clearly see that the text values have been converted into numeric indices.

##### Step 2: Verify All Categorical Columns

In [0]:
from pyspark.ml import Pipeline

index_pipeline = Pipeline(stages=indexers)

index_model = index_pipeline.fit(feature_df)

indexed_df = index_model.transform(feature_df)

display(indexed_df)

In [0]:
#Now checking for Schema for indexed_df

indexed_df.printSchema()

In [0]:
display(
    indexed_df.select(
        "product_id",
        "product_id_index",
        "category",
        "category_index",
        "gender",
        "gender_index",
        "city",
        "city_index",
        "state",
        "state_index"
    )
)

In [0]:
 #Instead of displaying the entire table, we can inspect each categorical feature individually along with its indexed values
 
# Product ID Mapping
display(
    indexed_df.select("product_id", "product_id_index")
              .distinct()
              .orderBy("product_id_index")
)

# Category Mapping
display(
    indexed_df.select("category", "category_index")
              .distinct()
              .orderBy("category_index")
)

# Gender Mapping
display(
    indexed_df.select("gender", "gender_index")
              .distinct()
              .orderBy("gender_index")
)

# City Mapping
display(
    indexed_df.select("city", "city_index")
              .distinct()
              .orderBy("city_index")
)

# State Mapping
display(
    indexed_df.select("state", "state_index")
              .distinct()
              .orderBy("state_index")
)

We can Observe that,

-  StringIndexer works correctly.
-  OneHotEncoder works correctly.

In [0]:
display(feature_df)

##### Step 3: Verify OneHotEncoder

In [0]:
encoder_model = encoder.fit(indexed_df)

encoded_df = encoder_model.transform(indexed_df)

display(encoded_df)

In [0]:
# Product ID Encoding
display(
    encoded_df.select(
        "product_id",
        "product_id_index",
        "product_id_encoded"
    ).distinct()
)

# Category Encoding
display(
    encoded_df.select(
        "category",
        "category_index",
        "category_encoded"
    ).distinct()
)

# Gender Encoding
display(
    encoded_df.select(
        "gender",
        "gender_index",
        "gender_encoded"
    ).distinct()
)

# City Encoding
display(
    encoded_df.select(
        "city",
        "city_index",
        "city_encoded"
    ).distinct()
)

# State Encoding
display(
    encoded_df.select(
        "state",
        "state_index",
        "state_encoded"
    ).distinct()
)

#### VectorAssembler

- VectorAssembler is a Spark ML transformer that combines multiple feature columns into a single vector column called features.

- Machine learning algorithms in Spark expect all input features to be in one vector column, not in separate columns

Before VectorAssembler


- Quantity	UnitPrice	Product_OHE	Gender_OHE
- 2	500	[0,1,0]	[1,0]

↓

After VectorAssembler
- features
[2,500,0,1,0,1,0]

All feature columns are merged into a single vector.

Why not keep separate columns?

Spark ML estimators (such as Linear Regression, Random Forest, GBT, Logistic Regression, etc.) are designed to read:

- One feature column → features
- One target column → e.g., Total_revenue

Keeping features in separate columns would cause the model training to fail because Spark ML expects a vector input.

#### Start Implement Vector Assembler
#####Step 1 – Define Input Columns

In [0]:
# Encoded categorical columns

encoded_cols = ['product_id_encoded', 'category_encoded', 'gender_encoded', 'city_encoded', 'state_encoded']

# Numeric columns

numeric_cols = ['Month', 'Quarter', 'DayOfWeek', 'DayOfMonth', 'WeekOfYear', 'Total_quantity', 'Max_price']

In [0]:
# Combine Them Both 

features_cols = encoded_cols + numeric_cols

##### Step 2 – Create the VectorAssembler

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=features_cols,
    outputCol='features',
    handleInvalid='keep'
)

##### Step 3 – Build a Temporary Pipeline for Verification

In [0]:
from pyspark.ml import Pipeline

pipeline = Pipeline(
    stages=indexers + [encoder, assembler]
)


##### Step 4 – Fit the Pipeline

In [0]:
# Clear previous model references to free ML cache
del category_model, index_model, encoder_model
import gc
gc.collect()

pipeline_model = pipeline.fit(feature_df)

##### Step 5 – Transform the Dataset

In [0]:
final_df = assembler.transform(encoded_df)

In [0]:
display(final_df)


#### Step 6 – Verify the Features Column

In [0]:
display(
    final_df.select(
        "features",
        "Total_revenue"
    )
)

##### Step 7 – Verify the Schema

In [0]:
final_df.printSchema()

In [0]:
final_df.write.format('delta').mode('overwrite').saveAsTable('retail_project.gold.final_ml_sales_data')

In [0]:
display(final_df)

In [0]:
print('Rows :', final_df.count())
print('Columns :', len(final_df.columns))

In [0]:
display(feature_df)

In [0]:
feature_df.write.format('delta').mode('overwrite').saveAsTable('retail_project.gold.ml_model_sales_data')